In [1]:
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_base")

text = "Heinz Cream of Tomato Soup 400g"

labels = ["brand"]
model.predict_entities(text, labels)

/Users/omkar/Documents/SmartShop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/omkar/Documents/SmartShop/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [01:02<00:00, 15.63s/it]


[{'start': 0,
  'end': 5,
  'text': 'Heinz',
  'label': 'brand',
  'score': 0.6728764176368713}]

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp("Heinz Cream of ")

for ent in doc.ents:
    print(ent.text, ent.label_)

Heinz Cream of ORG


In [4]:
import pandas as pd
df = pd.read_csv("/Users/omkar/Documents/SmartShop/app/sains_food_cupboard.csv")

In [22]:
product_names = df[df['own_brand']==False]['names'].to_list()

In [23]:
model.predict_entities(product_names[0], labels)


[{'start': 0,
  'end': 16,
  'text': 'Maryland Cookies',
  'label': 'brand',
  'score': 0.7031998634338379},
 {'start': 17,
  'end': 37,
  'text': 'Chocolate Chip Minis',
  'label': 'brand',
  'score': 0.6647108197212219}]

In [55]:
diff_brands = df[df['own_brand'] == False][['names']]


In [59]:
diff_brands = pd.DataFrame(diff_brands['names'].unique(),columns=['names'])

In [20]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"

In [39]:
import os
import requests
import json

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

import re
import json

def safe_parse(content):
    # remove ```json ... ```
    match = re.search(r"\{.*\}", content, re.S)
    
    if match:
        return json.loads(match.group())
    
    return {}

def extract_brands_router(df, text_col="names", batch_size=50):
    
    df = df.copy()
    
    if "Brand" not in df.columns:
        df["Brand"] = None
    
    mask = df["Brand"].isna()
    sub_df = df[mask].reset_index()
    
    for i in range(0, len(sub_df), batch_size):
        batch = sub_df.iloc[i:i+batch_size]
        
        items = {
            str(idx): name
            for idx, name in zip(batch["index"], batch[text_col])
        }
        
        prompt = f"""
Extract brand names from product list.

Return ONLY JSON:
id -> brand

Rules:
- No explanation
- If unknown: "Unknown"

Products:
{json.dumps(items)}
"""
        
        payload = {
            "model": "google/gemma-4-31B-it",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
        
        try:
            response = requests.post(API_URL, headers=headers, json=payload)
            result = response.json()
            print(result)
            
            content = result["choices"][0]["message"]["content"]
            
            parsed = safe_parse(content)
            
            for idx, brand in parsed.items():
                df.at[int(idx), "Brand"] = brand
        
        except Exception as e:
            print(f"Batch {i//batch_size + 1} failed:", e)
        
        print(f"Processed batch {i//batch_size + 1}")
    
    return df

In [40]:
df_result = extract_brands_router(diff_brands.head(50), text_col="names")

{'id': '4cfce7a60f55579bdf8bbe8ac9fa7069', 'object': 'chat.completion', 'created': 1776782003, 'model': 'google/gemma-4-31b-it', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '```json\n{\n  "0": "Maryland",\n  "1": "Weetabix",\n  "2": "Walker\'s",\n  "8": "Stamford Street Co.",\n  "12": "Cravendale",\n  "24": "JS",\n  "26": "Heinz",\n  "28": "Walkers",\n  "30": "Napolina",\n  "31": "Napolina",\n  "32": "Oatly",\n  "33": "Stamford Street Co.",\n  "34": "Heinz",\n  "42": "Alpro",\n  "45": "KTC",\n  "46": "Cravendale",\n  "47": "Walkers",\n  "48": "McVitie\'s",\n  "49": "Heinz",\n  "50": "KTC",\n  "51": "Princes",\n  "55": "Stamford Street Co.",\n  "57": "Walkers",\n  "60": "Stamford Street Co.",\n  "61": "Hula Hoops",\n  "62": "Jacob\'s",\n  "63": "Batchelors",\n  "65": "Pom-Bear",\n  "66": "McVitie\'s",\n  "68": "Birds Eye",\n  "69": "KitKat",\n  "70": "Walkers",\n  "71": "Diet Coke",\n  "72": "Cadbury",\n  "73": "Tilda",\n  "74": "Alpro",\n  "79": "Cadbury",\n  "

In [65]:
df_result['names']

0               Maryland Cookies Chocolate Chip Minis x6
1                                    Weetabix Cereal x24
2                   Walker's Shortbread Fingers x10 160g
8      Stamford Street Co. Chopped Tomatoes in Tomato...
12     Cravendale Filtered Fresh Semi Skimmed Milk 2L...
24                                 JS Double Cream 300ml
26     Heinz Baked Beans in a Rich Tomato Sauce 4 x 415g
28           Walkers Ready Salted Multipack Crisps 6x25g
30                      Napolina Chopped Tomatoes 4x400g
31                        Napolina Chopped Tomatoes 400g
32          Oatly Oat Drink Barista Edition Long Life 1L
33     Stamford Street Co. Baked Beans in Tomato Sauc...
34         Heinz Baked Beans in a Rich Tomato Sauce 415g
42             Alpro Almond No Sugars Long Life Drink 1L
45                             KTC Chopped Tomatoes 400g
46     Cravendale Filtered Fresh Whole Milk 2L Freshe...
47         Walkers Cheese & Onion Multipack Crisps 6x25g
48     McVitie's Milk Chocolate

In [66]:
import os
import requests
import json
import re
import time

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

# -----------------------------
# SAFE JSON PARSER
# -----------------------------
def safe_parse(content):
    try:
        match = re.search(r"\{[\s\S]*\}", content)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {}

# -----------------------------
# SAFE API CALL WITH RETRY
# -----------------------------
def call_llm(payload, retries=2, wait=2):
    for attempt in range(retries):
        try:
            response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
            result = response.json()

            # handle HF errors
            if "choices" not in result:
                print("HF error:", result)
                time.sleep(wait)
                continue

            return result

        except Exception as e:
            print(f"API error (attempt {attempt+1}):", e)
            time.sleep(wait)

    return None

# -----------------------------
# MAIN BATCH FUNCTION
# -----------------------------
def extract_brands_router(df, text_col="names", batch_size=50):
    
    df = df.copy()

    if "Brand" not in df.columns:
        df["Brand"] = None

    mask = df["Brand"].isna()
    sub_df = df[mask].reset_index(drop=False)

    total_batches = (len(sub_df) // batch_size) + 1

    for i in range(0, len(sub_df), batch_size):

        batch = sub_df.iloc[i:i+batch_size]

        items = {
            str(idx): name
            for idx, name in zip(batch["index"], batch[text_col])
        }

        prompt = f"""
Extract brand names from product list.

Return ONLY valid JSON:
id -> brand

Rules:
- No explanation
- If unknown: "Unknown"

Products:
{json.dumps(items)}
"""

        payload = {
            "model": "google/gemma-4-31B-it",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }

        result = call_llm(payload)

        if result is None:
            print(f"Batch {i//batch_size + 1} failed completely")
            continue

        content = result["choices"][0]["message"]["content"]
        parsed = safe_parse(content)

        # write back safely
        for idx, brand in parsed.items():
            try:
                df.at[int(idx), "Brand"] = brand
            except Exception:
                continue

        print(f"Processed batch {i//batch_size + 1} / {total_batches}")

        # small delay to avoid rate limits
        time.sleep(0.3)

    return df

In [67]:
df_result = extract_brands_router(diff_brands, text_col="names", batch_size=50)

API error (attempt 1): HTTPSConnectionPool(host='router.huggingface.co', port=443): Read timed out. (read timeout=60)
Processed batch 1 / 126
Processed batch 2 / 126
Processed batch 3 / 126
Processed batch 4 / 126
Processed batch 5 / 126
Processed batch 6 / 126
Processed batch 7 / 126
Processed batch 8 / 126
Processed batch 9 / 126
Processed batch 10 / 126
Processed batch 11 / 126
API error (attempt 1): HTTPSConnectionPool(host='router.huggingface.co', port=443): Read timed out. (read timeout=60)
Processed batch 12 / 126
Processed batch 13 / 126
Processed batch 14 / 126
Processed batch 15 / 126
Processed batch 16 / 126
Processed batch 17 / 126
Processed batch 18 / 126
Processed batch 19 / 126
Processed batch 20 / 126
Processed batch 21 / 126
Processed batch 22 / 126
Processed batch 23 / 126
Processed batch 24 / 126
Processed batch 25 / 126
Processed batch 26 / 126
Processed batch 27 / 126
Processed batch 28 / 126
Processed batch 29 / 126
Processed batch 30 / 126
API error (attempt 1): 

In [50]:
len(diff_brands) == diff_brands.nunique

False

In [73]:
df_result.to_csv("llm_master_brand.csv")

In [69]:
diff_brands.nunique

<bound method DataFrame.nunique of                                                   names
0              Maryland Cookies Chocolate Chip Minis x6
1                                   Weetabix Cereal x24
2                  Walker's Shortbread Fingers x10 160g
3     Stamford Street Co. Chopped Tomatoes in Tomato...
4     Cravendale Filtered Fresh Semi Skimmed Milk 2L...
...                                                 ...
6257  Fruit Shoot Summer Fruits Kids Juice Drink 8x2...
6258                   Cadbury Creme Egg 5 x 40g (200g)
6259             Kind Chocolate Chip Cashew Bars 12x40g
6260  Merchant Gourmet Fragrant Lentil Madras Curry ...
6261             Walkers Marmite Multipack Crisps 6x25g

[6262 rows x 1 columns]>

In [79]:
df_result['Brand']

0                  Maryland
1                  Weetabix
2                  Walker's
3       Stamford Street Co.
4                Cravendale
               ...         
6257            Fruit Shoot
6258                Cadbury
6259                   Kind
6260       Merchant Gourmet
6261                Walkers
Name: Brand, Length: 6262, dtype: object

In [84]:
brands_master = set(df_result['Brand'].str.lower().str.strip().str.replace(r'\s+', ' ', regex=True))

In [91]:
import pandas as pd
import re


def map_brands_to_products(df, brands_list, text_col="names"):
    
    df = df.copy()
    
    # -----------------------------
    # CLEAN BRAND LIST (IMPORTANT FIX)
    # -----------------------------
    brands = [
        str(b).strip()
        for b in brands_list
        if pd.notna(b) and str(b).strip() != ""
    ]
    
    # normalize product names
    names = df[text_col].fillna("").str.lower()
    
    # sort brands by length (longest first)
    brands = sorted(brands, key=len, reverse=True)
    
    # escape brands for regex
    brands_escaped = [re.escape(b.lower()) for b in brands]
    
    # build regex pattern
    pattern = r'\b(' + '|'.join(brands_escaped) + r')\b'
    
    # extract brand
    df["Brand"] = names.str.extract(pattern, expand=False)
    
    # format nicely
    df["Brand"] = df["Brand"].str.title()
    
    # fallback
    df["Brand"] = df["Brand"].fillna("Unknown")
    
    return df

In [92]:
map_brands_to_products(diff_brands, brands_master, text_col="names")

,names,Brand
0,Maryland Cookies Chocolate Chip Minis x6,Maryland Cookies
1,Weetabix Cereal x24,Weetabix
2,Walker's Shortbread Fingers x10 160g,Walker'S Shortbread
3,Stamford Street Co. Chopped Tomatoes in Tomato...,Unknown
4,Cravendale Filtered Fresh Semi Skimmed Milk 2L...,Cravendale
...,...,...
6257,Fruit Shoot Summer Fruits Kids Juice Drink 8x2...,Fruit Shoot
6258,Cadbury Creme Egg 5 x 40g (200g),Cadbury
6259,Kind Chocolate Chip Cashew Bars 12x40g,Kind
6260,Merchant Gourmet Fragrant Lentil Madras Curry ...,Merchant Gourmet


In [97]:
"stamford" in brands_master

False

In [98]:
df_result

,names,Brand
0,Maryland Cookies Chocolate Chip Minis x6,Maryland
1,Weetabix Cereal x24,Weetabix
2,Walker's Shortbread Fingers x10 160g,Walker's
3,Stamford Street Co. Chopped Tomatoes in Tomato...,Stamford Street Co.
4,Cravendale Filtered Fresh Semi Skimmed Milk 2L...,Cravendale
...,...,...
6257,Fruit Shoot Summer Fruits Kids Juice Drink 8x2...,Fruit Shoot
6258,Cadbury Creme Egg 5 x 40g (200g),Cadbury
6259,Kind Chocolate Chip Cashew Bars 12x40g,Kind
6260,Merchant Gourmet Fragrant Lentil Madras Curry ...,Merchant Gourmet


In [99]:
df_result['Brand'].value_counts()

Brand
Cadbury          232
Heinz            144
Walkers           88
Kellogg's         88
Twinings          81
                ... 
Kitkat             1
Oggs               1
Ben & Jerry's      1
Pulsin             1
J-Basket           1
Name: count, Length: 1046, dtype: int64

In [100]:
def normalize_brands(brands_list):
    import pandas as pd
    import re
    
    clean_map = {}
    
    for b in brands_list:
        if pd.isna(b):
            continue
        
        original = str(b).strip()
        
        # normalize
        norm = original.lower()
        norm = re.sub(r"[’']", "", norm)     # remove apostrophes
        norm = re.sub(r"[^a-z0-9\s]", " ", norm)  # remove special chars
        norm = re.sub(r"\s+", " ", norm).strip()  # remove extra spaces
        
        # keep first clean version as canonical
        if norm not in clean_map:
            clean_map[norm] = original
    
    return clean_map

In [102]:
brand_map = normalize_brands(df_result['Brand'].unique())

In [104]:
def map_brands_normalized(df, brand_map, text_col="names"):
    import re
    
    df = df.copy()
    
    # normalize product names
    names = df[text_col].fillna("").str.lower()
    names = names.str.replace(r"[’']", "", regex=True)
    names = names.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    names = names.str.replace(r"\s+", " ", regex=True).str.strip()
    
    # sort normalized brands
    brands = sorted(brand_map.keys(), key=len, reverse=True)
    
    pattern = r'\b(' + '|'.join(map(re.escape, brands)) + r')\b'
    
    # extract normalized match
    df["Brand_norm"] = names.str.extract(pattern, expand=False)
    
    # map back to original brand
    df["Brand"] = df["Brand_norm"].map(brand_map)
    
    # fallback
    df["Brand"] = df["Brand"].fillna("Unknown")
    
    return df.drop(columns=["Brand_norm"])

In [106]:
final_df = map_brands_normalized(diff_brands, brand_map, text_col="names")

In [108]:
"Bassetts Vitamins" in final_df['Brand']

False

In [112]:
final_df[final_df["names"].str.startswith("bassetts", na=False)]

,names,Brand


In [110]:
final_df

,names,Brand
0,Maryland Cookies Chocolate Chip Minis x6,Maryland Cookies
1,Weetabix Cereal x24,Weetabix
2,Walker's Shortbread Fingers x10 160g,Walker's Shortbread
3,Stamford Street Co. Chopped Tomatoes in Tomato...,Stamford Street Co.
4,Cravendale Filtered Fresh Semi Skimmed Milk 2L...,Cravendale
...,...,...
6257,Fruit Shoot Summer Fruits Kids Juice Drink 8x2...,Fruit Shoot
6258,Cadbury Creme Egg 5 x 40g (200g),Cadbury
6259,Kind Chocolate Chip Cashew Bars 12x40g,Kind
6260,Merchant Gourmet Fragrant Lentil Madras Curry ...,Merchant Gourmet
